# Experiment B1 — fixed-$D$ batch-size sweep with noisy-Nesterov batch exponent

This notebook fixes a total scale at
\[
D=2^{15},\qquad K=\lfloor D/B\rfloor,
\]
and sweeps powers-of-two batch sizes. The vertical line for Nesterov uses the
largest batch exponent from the noisy setting in the screenshot:

\[
a_{\max}^{\mathrm{NAG,noisy}}=
\begin{cases}
\dfrac{2s}{s+2-1/\beta}, & 0<s\le (2-1/\beta)/2,\\[4pt]
\dfrac{2s+1/\beta}{2s+1+1/\beta}, & s\ge (2-1/\beta)/2.
\end{cases}
\]

The notebook still uses only one scalar `S_VALUE`; the branch is selected
automatically. HB and Nesterov use the corresponding nominal momentum exponent
from the same noisy-Nesterov schedule, while SGD keeps $\rho=0$.

The code uses chunked minibatch gradients so large batches do not require materializing a full `(B, DIM)` matrix at once.


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

BETA = 2.0
SIGMA = 1.0
DIM = 4096
D_FIXED = 2 ** 15
ETA_FIXED = 0.05
C_RHO = 1.0
Y_GAP_FRACTION = 0.05
CHUNK = 2048
N_SEEDS = 3
LOAD_CACHE = True
SAVE_CACHE = True

# Single source-smoothness parameter.
# For BETA=2: S_VALUE < 0.75 uses branch 1; S_VALUE >= 0.75 uses branch 2.
S_VALUE = 0.50

ALGOS = {
    'HB': 'hb',
    'Nesterov': 'nag',
    'SGD': 'sgd',
}

B_EXPONENTS = list(range(3, 15))
B_LIST = [2 ** e for e in B_EXPONENTS if 2 ** e <= D_FIXED]
RESULT_DIR = Path('results')
RESULT_DIR.mkdir(exist_ok=True)

q_value = 2.0 - 1.0 / BETA
print(f'beta={BETA:g}, q={q_value:.3f}, q/2={0.5 * q_value:.3f}, sigma={SIGMA:g}, dim={DIM}, D=2^{int(np.log2(D_FIXED))}, seeds={N_SEEDS}')
print(f's={S_VALUE:g}')
print(f'batch grid: 2^{B_EXPONENTS[0]} ... 2^{B_EXPONENTS[-1]} ({len(B_LIST)} values)')


In [ ]:
def build_problem(dim=DIM, beta=BETA, s=1.0):
    """Return (lam, theta_star) for the diagonal linear-regression problem."""
    j = np.arange(1, dim + 1, dtype=np.float64)
    lam = j ** (-beta)
    var_star = j ** (-(1.0 + (s - 1.0) * beta))
    theta_star = np.sqrt(var_star)
    return lam, theta_star


def excess_loss(delta, lam):
    """Exact population excess loss E = 1/2 * sum_j lam_j * delta_j^2."""
    return 0.5 * np.sum(lam * delta * delta)


def grad_delta_chunked(delta, B, lam, sigma=SIGMA, rng=None, chunk=CHUNK):
    """Exact stochastic minibatch gradient without materializing a full (B, DIM) batch."""
    rng = rng or np.random.default_rng()
    d = lam.shape[0]
    sqrt_lam = np.sqrt(lam)
    grad = np.zeros(d, dtype=np.float64)
    remaining = int(B)
    while remaining > 0:
        b = min(chunk, remaining)
        X = rng.standard_normal((b, d)) * sqrt_lam
        eps = rng.standard_normal(b) * sigma
        resid = X @ delta - eps
        grad += X.T @ resid
        remaining -= b
    return grad / B


def run_optimizer(algo, B, K, eta, rho, lam, theta_star, sigma=SIGMA, rng=None, record_curve=False):
    """Run one of {'hb','nag','sgd'} for K steps of batch size B."""
    rng = rng or np.random.default_rng()
    if algo == 'sgd':
        rho = 0.0
    delta = -theta_star.copy()
    delta_prev = delta.copy()
    curve = [excess_loss(delta, lam)] if record_curve else None
    for _ in range(int(K)):
        momentum = delta - delta_prev
        eval_delta = delta + rho * momentum if algo == 'nag' else delta
        g = grad_delta_chunked(eval_delta, B, lam, sigma=sigma, rng=rng)
        delta_new = delta - eta * g + rho * momentum
        delta_prev, delta = delta, delta_new
        if record_curve:
            curve.append(excess_loss(delta, lam))
    final_E = excess_loss(delta, lam)
    return final_E, np.array(curve) if record_curve else None


def q_noisy(beta=BETA):
    return 2.0 - 1.0 / beta


def noisy_nag_scaling(s, beta=BETA):
    q = q_noisy(beta)
    if s <= 0.0:
        raise ValueError(f's must be positive, got {s}')

    if s < 0.5 * q:
        branch = '0 < s < q/2'
        x = s / (s + q)
        a_max = 2.0 * s / (s + q)
        a = a_max
        y = 3.0 * x
    else:
        branch = 's >= q/2'
        x = (s - 0.5 * (1.0 - 1.0 / beta)) / (s + 0.5 * (1.0 + 1.0 / beta))
        a_max = 0.5 * (1.0 + x)
        y = 1.0 - Y_GAP_FRACTION * (1.0 - x)
        a = 0.5 * (x + y)

    b = y - a
    return {'branch': branch, 'x': x, 'a': a, 'a_max_nag_noisy': a_max, 'y': y, 'b': b}


def alpha_hb(s, beta=BETA):
    return min(s, q_noisy(beta))


def amax_nag_noisy(s, beta=BETA):
    """Largest noisy-Nesterov batch exponent from the screenshot."""
    return noisy_nag_scaling(s, beta)['a_max_nag_noisy']


def predicted_batch_exponents(s, beta=BETA):
    """Theory-guided useful-batch exponents a=log(B)/log(D)."""
    return {
        'SGD': s / (s + 1.0),
        'HB': min((2.0 * s + 1.0) / (2.0 * s + 2.0), (s + alpha_hb(s, beta)) / (s + 1.0)),
        'Nesterov': amax_nag_noisy(s, beta),
    }


def clipped_rho(one_minus_rho):
    one_minus_rho = float(np.clip(one_minus_rho, 1e-5, 1.0))
    return float(np.clip(1.0 - one_minus_rho, 0.0, 0.99999))


def fixed_D_nominal_rho(s, D=D_FIXED, c_rho=C_RHO):
    scaling = noisy_nag_scaling(s)
    return clipped_rho(c_rho * D ** (-scaling['b']))


def cache_key(prefix):
    s_tag = f's{S_VALUE:g}'.replace('.', '_')
    return RESULT_DIR / f'{prefix}_noisy_nag_{s_tag}_dim{DIM}_D{D_FIXED}_beta{BETA:g}_sigma{SIGMA:g}_seeds{N_SEEDS}.npz'


SCALING = noisy_nag_scaling(S_VALUE)
print(f"branch={SCALING['branch']}, a={SCALING['a']:.4f}, "
      f"a_max={SCALING['a_max_nag_noisy']:.4f}, y={SCALING['y']:.4f}, b={SCALING['b']:.4f}")


## Run the fixed-momentum sweep

For small `B`, `K=D/B` is very large. If you are debugging, reduce `DIM`, `N_SEEDS`, or narrow `B_EXPONENTS`; for the paper run, keep `D_FIXED=2**20` and the full grid.


In [ ]:
cache_path = cache_key('batch_fixed_momentum')

if LOAD_CACHE and cache_path.exists():
    BATCH_RESULTS = np.load(cache_path, allow_pickle=True)['results'].item()
    print(f'loaded cached results from {cache_path}')
else:
    lam, theta_star = build_problem(s=S_VALUE)
    rho_momentum = fixed_D_nominal_rho(S_VALUE)
    print(f"=== s={S_VALUE:.3f}  fixed rho(HB/NAG)={rho_momentum:.6f} ===")

    BATCH_RESULTS = {'B': B_LIST, 'rho_momentum': rho_momentum}
    for name in ALGOS:
        BATCH_RESULTS[name] = {'mean': [], 'std': [], 'all': []}

    for B in B_LIST:
        K = max(1, D_FIXED // B)
        print(f'B=2^{int(np.log2(B)):<2d} ({B:6d}), K={K:7d}')
        for name, algo in ALGOS.items():
            rho = 0.0 if algo == 'sgd' else rho_momentum
            vals = []
            for seed in range(N_SEEDS):
                rng = np.random.default_rng(1_000_000 + 100_000 * seed + 1000 * int(100 * S_VALUE) + B)
                final_E, _ = run_optimizer(algo, B, K, ETA_FIXED, rho, lam, theta_star, rng=rng)
                vals.append(final_E)
            vals = np.array(vals)
            BATCH_RESULTS[name]['mean'].append(float(vals.mean()))
            BATCH_RESULTS[name]['std'].append(float(vals.std()))
            BATCH_RESULTS[name]['all'].append(vals.tolist())
            print(f'  {name:9s} E_K={vals.mean():.3e} +/- {vals.std():.1e}  rho={rho:.5f}')

    if SAVE_CACHE:
        np.savez_compressed(cache_path, results=BATCH_RESULTS)
        print(f'saved results to {cache_path}')


## Plot: final excess loss versus batch size


In [ ]:
def plot_batch_sweep(results, title_suffix):
    colors = {'HB': 'tab:blue', 'Nesterov': 'tab:orange', 'SGD': 'tab:green'}
    markers = {'HB': 'o', 'Nesterov': 's', 'SGD': '^'}
    linestyles = {'HB': ':', 'Nesterov': '--', 'SGD': '-.'}

    fig, ax = plt.subplots(figsize=(8.0, 5.4))
    B_arr = np.array(results['B'], dtype=float)
    pred = predicted_batch_exponents(S_VALUE)

    for name in ALGOS:
        mean = np.array(results[name]['mean'])
        std = np.array(results[name]['std'])
        ax.errorbar(B_arr, mean, yerr=std, marker=markers[name], capsize=3,
                    color=colors[name], label=name)

    ymax = max(np.nanmax(results[name]['mean']) for name in ALGOS)
    ymin = min(np.nanmin(results[name]['mean']) for name in ALGOS)
    for name, a_pred in pred.items():
        B_pred = D_FIXED ** a_pred
        if B_arr[0] <= B_pred <= B_arr[-1]:
            ax.axvline(B_pred, color=colors[name], linestyle=linestyles[name], alpha=0.75,
                       label=rf'{name} predicted $B\approx 2^{{{np.log2(B_pred):.1f}}}$')

    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('batch size  $B$')
    ax.set_ylabel(r'final excess loss  $\mathcal{E}_K$')
    ax.set_title(f"s={S_VALUE:.2f}  {title_suffix}")
    ax.grid(True, which='both', color=(0, 0, 0, 0.25))
    ax.legend(fontsize=8)
    ax.set_ylim(max(ymin * 0.5, 1e-12), ymax * 2.0)

    plt.tight_layout()
    plt.show()

plot_batch_sweep(BATCH_RESULTS, 'fixed momentum')


## Notes

Expected behavior is an elbow or U-shape: small batches have a larger stochastic noise floor, while very large batches have too few optimization steps. Momentum should move the useful-batch region to the right compared with SGD when the constants are well tuned.
